# RQ3 Preliminary Test: PHO Burden vs CIHI Access Join

**CIND 820 Big Data Analytics Project, Milestone 2: Architecture and Data Audit**

Mamkon Mercy Oyeleke | Student Number: 501279489 | Toronto Metropolitan University

**Data source:** Outputs of 01_pho_eda.py and 02_cihi_eda.py (uses the same raw files in ../data/raw/)

**Outputs:** `../outputs/figures/fig11_rq3_burden_vs_access.png` and `../outputs/tables/rq3_burden_vs_access.csv`

Run the setup cell below first. It clones this repository when running in Google Colab, so the relative `../data/raw/` and `../outputs/` paths used throughout this notebook resolve correctly.

## Setup

In [ ]:
import os, sys

# If running in Google Colab, clone the repo so relative data/output paths resolve correctly
if 'google.colab' in sys.modules:
    if not os.path.exists('Architecture_and_Data_Audit'):
        os.system('git clone https://github.com/mamkon/Architecture_and_Data_Audit.git')
    os.chdir('Architecture_and_Data_Audit/notebooks')

print("Working directory:", os.getcwd())

## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 150,
})

## PHO side: composite burden score by region, 2023

In [ ]:
pho = pd.read_excel('../data/raw/PHO_Chronic_Disease_Inc_Prev_Snapshot_2014_2023.xlsx', skiprows=2)

regions = ['Central East', 'Central West', 'East', 'North East', 'North West',
           'South West', 'Toronto', 'Ontario']
indicator_map = {
    'Prevalence of asthma': 'Asthma',
    'Prevalence of COPD in adults 20+': 'COPD',
    'Prevalence of diabetes in adults 20+': 'Diabetes',
    'Prevalence of hypertension in adults 20+': 'Hypertension',
}

pho_sub = pho[(pho['Measure'] == 'Age-standardized rate (both sexes)') &
              (pho['Year'] == 2023) &
              (pho['Geography'].isin(regions)) &
              (pho['Indicator'].str.contains('Prevalence'))].copy()
pho_sub['Ind'] = pho_sub['Indicator'].map(indicator_map)
piv = pho_sub.pivot_table(index='Geography', columns='Ind', values='Rate')
piv = piv.drop('Ontario')

# Normalise and build composite burden score (consistent with PHO EDA methodology)
norm = (piv - piv.min()) / (piv.max() - piv.min())
piv['Burden Score'] = norm.mean(axis=1)

print("PHO composite burden score by region (2023):")
print(piv[['Burden Score']].round(3).to_string())

# Map PHO's 7 regions onto CIHI's 6 regions
pho_to_cihi = {
    'Central East': 'Central Region', 'Central West': 'Central Region',
    'East': 'East Region', 'North East': 'North East Region',
    'North West': 'North West Region', 'South West': 'West Region',
    'Toronto': 'Toronto Region',
}
piv['CIHI_Region'] = piv.index.map(pho_to_cihi)
burden_by_cihi = piv.groupby('CIHI_Region')['Burden Score'].mean().round(3)

print("\nPHO composite burden score, mapped to CIHI's 6 Ontario regions (2023):")
print(burden_by_cihi.sort_values(ascending=False).to_string())

## CIHI side: median Hip+Knee replacement wait, by region, 2024

In [ ]:
cihi = pd.read_excel('../data/raw/CIHI_Wait_Times_Priority_Procedures_2025.xlsx',
                      sheet_name='Table 1', header=1)
cihi = cihi[cihi['Province'].notna()]
on_reg = cihi[(cihi['Province'] == 'Ontario') & (cihi['Reporting level'] == 'Regional') &
              (cihi['Data year'] == 2024) & (cihi['Metric'] == '50th Percentile')]
wait_piv = on_reg.pivot_table(index='Region', columns='Indicator', values='Indicator result')
wait_piv['Avg Wait (Hip+Knee, days)'] = wait_piv[['Hip Replacement', 'Knee Replacement']].mean(axis=1)

print("\nCIHI median wait times by region (2024):")
print(wait_piv.round(1).to_string())

## JOIN

In [ ]:
joined = pd.DataFrame({
    'Burden Score': burden_by_cihi,
    'Avg Wait Hip+Knee (days)': wait_piv['Avg Wait (Hip+Knee, days)'],
}).sort_values('Burden Score', ascending=False)

print("\n" + "=" * 60)
print("RQ3 PRELIMINARY JOIN - Burden vs Access, by Ontario region")
print("=" * 60)
print(joined.round(2).to_string())

corr = joined['Burden Score'].corr(joined['Avg Wait Hip+Knee (days)'])
print(f"\nPearson correlation (burden vs wait time): r = {corr:.3f}")

joined.to_csv('../outputs/tables/rq3_burden_vs_access.csv')
print("\nSaved rq3_burden_vs_access.csv")

## VISUAL

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('RQ3 Preliminary Test - Chronic Disease Burden vs Access to Care, '
              'Ontario Regions (2023-2024)', fontsize=12, fontweight='bold')

joined_sorted = joined.sort_values('Burden Score')
axes[0].barh(joined_sorted.index, joined_sorted['Burden Score'], color='#D85A30', edgecolor='white')
axes[0].set_xlabel('Composite chronic disease burden score (2023)')
axes[0].set_title('Burden by region')

axes[1].scatter(joined['Burden Score'], joined['Avg Wait Hip+Knee (days)'], s=80, color='#185FA5')
for region, row in joined.iterrows():
    axes[1].annotate(region.replace(' Region', ''),
                      (row['Burden Score'], row['Avg Wait Hip+Knee (days)']),
                      fontsize=8, xytext=(5, 5), textcoords='offset points')
axes[1].set_xlabel('Composite burden score (2023)')
axes[1].set_ylabel('Avg median wait, Hip+Knee replacement, 2024 (days)')
axes[1].set_title(f'Burden vs wait time (r = {corr:.2f})')

plt.tight_layout()
plt.savefig('../outputs/figures/fig11_rq3_burden_vs_access.png', bbox_inches='tight')
plt.close()
print("Saved fig11_rq3_burden_vs_access.png")

print("""
INTERPRETATION:
North East Region stands out on both dimensions: highest composite chronic
disease burden (0.88, more than 1.5x the next-highest region) AND the
second-longest average wait time for hip/knee replacement (142 days).
Toronto Region sits at the opposite end on both measures (burden 0.31,
wait 48 days). The moderate positive correlation (r = 0.61) across just 6
regions is a preliminary signal, not a statistical claim - the sample size
is too small (n=6) for significance testing. This finding will be carried
into Milestone 3 as a candidate result requiring confirmation with a finer-
grained or multi-year analysis.
""")